# 테스트셋 분포 편향 평가

**목적**: 테스트셋의 독성/비독성 비율이 성능 지표에 미치는 영향 분석  
**사용 모델**: `model_best.joblib` (MACCS only + SVM, 기존 학습 완료 모델)  

| 테스트셋 | 구성 | 설명 |
|---|---|---|
| Stratified (기존) | 독성 26.8% / 비독성 73.2% | 원래 데이터 비율 유지 |
| All-Toxic | 독성 100% | 극단적 편향 (독성만) |
| All-Non-Toxic | 비독성 100% | 극단적 편향 (비독성만) |

## 1. 라이브러리 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    balanced_accuracy_score, roc_auc_score, accuracy_score,
    matthews_corrcoef, f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import seaborn as sns

def _set_korean_font():
    candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']
    available = {f.name for f in fm.fontManager.ttflist}
    for font in candidates:
        if font in available: return font
    return 'DejaVu Sans'
matplotlib.rcParams['font.family'] = _set_korean_font()
matplotlib.rcParams['axes.unicode_minus'] = False

# 원본 데이터 로드
df_desc = pd.read_csv('final_dataset_descriptors.csv')
maccs_cols = [c for c in df_desc.columns if c.startswith('MACCS_')]
X_maccs = df_desc[maccs_cols].values.astype(np.float32)
y_clf   = df_desc['label'].values.astype(int)

print(f'데이터 로드 완료: {df_desc.shape}')
print(f'독성(1): {y_clf.sum()}개  비독성(0): {(y_clf==0).sum()}개')

## 2. 기존 모델 로드 & 학습셋 재현

기존 노트북과 동일한 random_state 없이 split했으므로, **동일한 train index를 재현하기 위해 모델을 전체 데이터로 재학습**합니다.  
단, 테스트셋은 아래 3가지로 새로 구성합니다.

In [ ]:
# 저장된 모델 로드
clf_loaded = joblib.load('model_best.joblib')
model_clf  = clf_loaded['model']
m_saved    = clf_loaded['test_metrics']

print(f'모델: {clf_loaded["best_model_name"]} / {clf_loaded["best_descriptor"]}')
print(f'저장된 테스트셋 성능 (Stratified 20%)')
print(f'  Balanced Accuracy : {m_saved["balanced_accuracy"]:.4f}')
print(f'  ROC-AUC           : {m_saved["roc_auc"]:.4f}')
print(f'  MCC               : {m_saved["mcc"]:.4f}')
print(f'  F1                : {m_saved["f1"]:.4f}')

## 3. 편향 테스트셋 구성

- **기존 stratified split**: train_test_split으로 재현 (random_state 고정)
- **All-Toxic**: 테스트셋 중 독성 분자만
- **All-Non-Toxic**: 테스트셋 중 비독성 분자만

In [ ]:
# stratified split 재현 (random_state=42 고정)
idx = np.arange(len(y_clf))
idx_trainval, idx_test = train_test_split(idx, test_size=0.2, stratify=y_clf, random_state=42)

X_train = X_maccs[idx_trainval]
y_train = y_clf[idx_trainval]

X_test_strat = X_maccs[idx_test]
y_test_strat = y_clf[idx_test]

# All-Toxic: 테스트셋에서 독성(1)만
mask_toxic   = y_test_strat == 1
X_test_toxic = X_test_strat[mask_toxic]
y_test_toxic = y_test_strat[mask_toxic]

# All-Non-Toxic: 테스트셋에서 비독성(0)만
mask_nontox     = y_test_strat == 0
X_test_nontoxic = X_test_strat[mask_nontox]
y_test_nontoxic = y_test_strat[mask_nontox]

print(f'Train+Val       : {len(idx_trainval)}개  (독성 {y_train.sum()}, 비독성 {(y_train==0).sum()})')
print(f'Test Stratified : {len(y_test_strat)}개  (독성 {y_test_strat.sum()}, 비독성 {(y_test_strat==0).sum()})')
print(f'Test All-Toxic  : {len(y_test_toxic)}개  (독성 100%)')
print(f'Test All-NonTox : {len(y_test_nontoxic)}개  (비독성 100%)')

## 4. 모델 재학습 (Train+Val 전체)

저장된 모델과 동일한 하이퍼파라미터로 Train+Val 전체를 학습합니다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(C=1.0, kernel='rbf', class_weight='balanced', probability=True))
])
clf.fit(X_train, y_train)
print('재학습 완료')

## 5. 3가지 테스트셋 성능 비교

In [ ]:
def evaluate(clf, X, y, name):
    """테스트셋 평가 — ROC-AUC는 두 클래스가 모두 있을 때만 계산"""
    proba  = clf.predict_proba(X)[:, 1]
    y_pred = (proba >= 0.5).astype(int)

    bal_acc = balanced_accuracy_score(y, y_pred)
    acc     = accuracy_score(y, y_pred)
    f1      = f1_score(y, y_pred, zero_division=0)
    mcc     = matthews_corrcoef(y, y_pred)

    unique_classes = np.unique(y)
    if len(unique_classes) == 2:
        roc_auc = roc_auc_score(y, proba)
    else:
        roc_auc = float('nan')  # 단일 클래스이면 계산 불가

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')

    return {
        'TestSet'          : name,
        'N'                : len(y),
        'Toxic%'           : f'{y.mean()*100:.1f}%',
        'Balanced Accuracy': round(bal_acc, 4),
        'Accuracy'         : round(acc, 4),
        'ROC-AUC'          : round(roc_auc, 4) if not np.isnan(roc_auc) else 'N/A',
        'F1'               : round(f1, 4),
        'MCC'              : round(mcc, 4),
        'Sensitivity'      : round(sensitivity, 4) if not np.isnan(sensitivity) else 'N/A',
        'Specificity'      : round(specificity, 4) if not np.isnan(specificity) else 'N/A',
        '_proba'           : proba,
        '_y'               : y,
    }

res_strat   = evaluate(clf, X_test_strat,   y_test_strat,   'Stratified (26.8% toxic)')
res_toxic   = evaluate(clf, X_test_toxic,   y_test_toxic,   'All-Toxic (100%)')
res_nontox  = evaluate(clf, X_test_nontoxic,y_test_nontoxic,'All-Non-Toxic (0%)')

display_cols = ['TestSet','N','Toxic%','Balanced Accuracy','Accuracy',
                'ROC-AUC','F1','MCC','Sensitivity','Specificity']
results_df = pd.DataFrame([res_strat, res_toxic, res_nontox])[display_cols]
print(results_df.to_string(index=False))

## 6. 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('테스트셋 분포 편향에 따른 분류 성능 비교 (MACCS only + SVM)',
             fontsize=13, fontweight='bold')

labels = ['Stratified\n(26.8%)', 'All-Toxic\n(100%)', 'All-NonToxic\n(0%)']
colors = ['steelblue', 'tomato', 'seagreen']

# ── 왼쪽: 주요 지표 그룹 바차트 ───────────────────────────────────
metrics_to_plot = ['Balanced Accuracy', 'Accuracy', 'F1', 'MCC']
x = np.arange(len(metrics_to_plot))
w = 0.25

for i, (res, lbl, col) in enumerate(zip([res_strat, res_toxic, res_nontox], labels, colors)):
    vals = []
    for m in metrics_to_plot:
        v = res[m]
        vals.append(float(v) if v != 'N/A' else 0.0)
    bars = axes[0].bar(x + i*w - w, vals, w, label=lbl, color=col, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                     f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_to_plot, fontsize=10)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('성능 지표 비교', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.2)
axes[0].grid(axis='y', alpha=0.3)

# ── 가운데: Sensitivity / Specificity 비교 ───────────────────────
sens_vals = []
spec_vals = []
for res in [res_strat, res_toxic, res_nontox]:
    s = res['Sensitivity']
    p = res['Specificity']
    sens_vals.append(float(s) if s != 'N/A' else np.nan)
    spec_vals.append(float(p) if p != 'N/A' else np.nan)

x2 = np.arange(3)
b1 = axes[1].bar(x2 - 0.2, sens_vals, 0.38, label='Sensitivity', color='tomato', alpha=0.85, edgecolor='white')
b2 = axes[1].bar(x2 + 0.2, spec_vals, 0.38, label='Specificity', color='steelblue', alpha=0.85, edgecolor='white')
for bar in list(b1) + list(b2):
    h = bar.get_height()
    if not np.isnan(h):
        axes[1].text(bar.get_x()+bar.get_width()/2, h+0.01,
                     f'{h:.3f}', ha='center', va='bottom', fontsize=8)
    else:
        axes[1].text(bar.get_x()+bar.get_width()/2, 0.02,
                     'N/A', ha='center', va='bottom', fontsize=8, color='gray')

axes[1].set_xticks(x2)
axes[1].set_xticklabels(labels, fontsize=9)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Sensitivity / Specificity', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 1.2)
axes[1].grid(axis='y', alpha=0.3)

# ── 오른쪽: 예측 확률 분포 (violin) ─────────────────────────────
plot_data = []
for res, lbl in zip([res_strat, res_toxic, res_nontox], labels):
    for p in res['_proba']:
        plot_data.append({'TestSet': lbl.replace('\n', ' '), 'Prob_Toxic': p})
plot_df = pd.DataFrame(plot_data)

sns.violinplot(data=plot_df, x='TestSet', y='Prob_Toxic',
               palette=colors, ax=axes[2], inner='box', linewidth=1.2)
axes[2].axhline(0.5, color='gray', ls='--', lw=1.2, label='Threshold 0.5')
axes[2].set_xlabel('')
axes[2].set_ylabel('P(Toxic)', fontsize=11)
axes[2].set_title('예측 확률 분포', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig('bias_test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장: bias_test_evaluation.png')

## 7. 결과 해석

In [ ]:
strat_bal = res_strat['Balanced Accuracy']
toxic_acc = res_toxic['Accuracy']
nontox_acc = res_nontox['Accuracy']

print('=' * 60)
print('결과 요약 및 해석')
print('=' * 60)
print()
print('[1] Stratified 테스트셋 (기존, 독성 26.8%)')
print(f'    Balanced Accuracy = {strat_bal:.4f}  ROC-AUC = {res_strat["ROC-AUC"]}')
print('    → 학습 데이터와 동일한 분포 → 성능이 유리하게 측정될 수 있음')
print()
print('[2] All-Toxic 테스트셋 (독성 100%)')
print(f'    Balanced Accuracy = {res_toxic["Balanced Accuracy"]:.4f}  Accuracy = {toxic_acc:.4f}')
print(f'    Sensitivity = {res_toxic["Sensitivity"]}  (독성 분자 검출률)')
print('    → Specificity 계산 불가 (비독성 샘플 없음)')
print('    → 모델이 독성 분자만 있을 때 얼마나 잘 잡아내는지 평가')
print()
print('[3] All-Non-Toxic 테스트셋 (비독성 100%)')
print(f'    Balanced Accuracy = {res_nontox["Balanced Accuracy"]:.4f}  Accuracy = {nontox_acc:.4f}')
print(f'    Specificity = {res_nontox["Specificity"]}  (비독성 분자 정확 분류율)')
print('    → Sensitivity 계산 불가 (독성 샘플 없음)')
print('    → 모델이 비독성 분자를 잘못 독성으로 분류하는 비율 확인')
print()
print('[결론]')
print('    Stratified 테스트셋 성능이 편향된 테스트셋보다 높게 나타난다면,')
print('    테스트 분포가 성능 측정에 유리하게 작용했다고 볼 수 있음.')
print('    반대로 편향 테스트셋에서도 성능이 유지된다면,')
print('    모델이 분포 변화에 강건(robust)하다고 해석할 수 있음.')

# CSV 저장
results_df.to_csv('bias_test_results.csv', index=False, encoding='utf-8-sig')
print()
print('결과 저장: bias_test_results.csv')